# Is one epoch enough? Training-budget ablation for the LoRA extractor (Kaggle version)

Same experiment as `ablation_colab.ipynb`: a grid of **epochs (1, 2, 3) x training size (2,000 vs. the
full train split)**, each trained from a fresh base model and scored on a fixed sample of the
**validation** split. The best configuration is chosen there; the **test** set is used once, for the
selected configuration and the reference recipe (2,000 sentences, 1 epoch).

**Setup (once):**
1. A Kaggle account with a verified phone number (needed for GPU and internet access).
2. *File > Import notebook*, then paste this notebook's GitHub URL or upload the `.ipynb`.
3. In the notebook's right-hand panel: **Accelerator: GPU T4 x2** and **Internet: on**.

**Run it in the background:** click **Save Version > Save & Run All (Commit)**. The run continues with
the browser closed (up to 12 hours), and everything in `/kaggle/working` is kept as the version's output.

**Resuming:** every finished configuration is saved immediately. If a run stops before the end, open
the notebook, use **Add Input > Your Work** to attach this notebook's last version, and commit again.
Cell 3 copies the earlier checkpoints and adapters back, so only unfinished configurations are trained.

In [ ]:
# 1. GPU check. Use one GPU: the trainer would otherwise wrap the model across both T4s.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No GPU found. Settings > Accelerator > GPU T4 x2."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install the package from GitHub (needs Internet: on). Kaggle's CUDA build of torch is kept.
# Older torchao builds break recent peft; this project does not use torchao, so remove it.
%pip uninstall -q -y torchao
%pip install -q "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"
%pip install -q --force-reinstall --no-deps "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"

import clinical_lora

print("clinical_lora", clinical_lora.__version__)

In [ ]:
# 3. Resume: copy checkpoints and adapters from an earlier version attached as input (if any)
import shutil
from pathlib import Path

ROOT = Path("/kaggle/working")
for prev in Path("/kaggle/input").glob("*"):
    for sub in ("results_ablation", "adapters"):
        src = prev / sub
        if src.is_dir():
            shutil.copytree(src, ROOT / sub, dirs_exist_ok=True)
            print("restored", src)
ckpts = sorted(p.name for p in (ROOT / "results_ablation" / "checkpoints").glob("*.json"))
print("checkpoints found:", ckpts or "none (fresh run)")

In [ ]:
# 4. Grid and sizes (edit here)
from clinical_lora.ablation import AblationConfig

EPOCHS = (1.0, 2.0, 3.0)
N_TRAIN = (2000, None)  # None = the full training split

acfg = AblationConfig(
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    epochs=EPOCHS,
    n_train=N_TRAIN,
    reference=(2000, 1.0),  # the recipe reported in the README
    n_val=300,  # validation sentences used for selection
    n_test=200,  # same 200 test sentences as the main notebook (seed 0)
    seed=0,
    out_dir=str(ROOT / "results_ablation"),
    adapter_root=str(ROOT / "adapters" / "ablation"),
    resume=True,
)
acfg

In [ ]:
# 5. Load BC5CDR once and estimate the run time
from clinical_lora.data import load_bc5cdr

splits = load_bc5cdr()
print({k: len(v) for k, v in splits.items()})

# The reported run trained 2,000 sentences for 1 epoch in 4.8 min on a T4; evaluating 300
# validation sentences adds roughly 2-3 min per configuration. Treat this as a rough guide;
# configurations restored in cell 3 are not retrained.
n_all = len(splits["train"])
units = sum((n or n_all) / 2000 * e for n in N_TRAIN for e in EPOCHS)
n_cfg = len(N_TRAIN) * len(EPOCHS)
print(f"{n_cfg} configurations, about {units * 4.8 + n_cfg * 2.5 + 10:.0f} minutes from scratch")

In [ ]:
# 6. Run the grid, select on validation, evaluate the winner and the reference once on test
from clinical_lora.ablation import run_ablation

results = run_ablation(acfg, splits=splits)
print("selected:", results["selected"], "| reference:", results["reference"])

In [ ]:
# 7. Report and plot. Download results_ablation/ablation.json from the version's Output tab
# and send it back to update the README.
from IPython.display import Image, Markdown, display

display(Markdown(open(f"{acfg.out_dir}/ablation.md").read()))
display(Image(f"{acfg.out_dir}/ablation_val_f1.png"))